# Lesson 15 Lab — TorchAO INT4 Weight-Only Quantization

**Puzzle:** Can a PyTorch-native INT4 conversion reduce storage and still lose on latency?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A PyTorch-native quantization API still depends on a precise package, ABI, hardware, and kernel combination. Conversion success, packed storage, numerical agreement, and latency are separate gates. Preserving a failed compatibility attempt is more useful than silently substituting a fake quantizer and calling it TorchAO.


## 0. Predict before running

1. Predict the evidence sequence required before comparing TorchAO INT4 latency with BF16.
2. Decide whether an installed `torchao` package is enough to claim native execution.
3. Explain how an ABI or auxiliary-kernel dependency can block an otherwise supported GPU.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

TorchAO conversion replaces or wraps eligible `Linear` weights with a packed tensor subclass/configuration. The Python module, packed storage, and selected matmul kernel are three inspectable layers.

- TorchAO replaces eligible modules according to a quantization configuration.
- Packed storage and executed operator evidence are distinct from a module label.
- Small batch and shape-specific overhead can outweigh lower memory traffic.


## 2. Derive the mechanism

INT4 weight-only compute conceptually reads packed codes and group scales while BF16 activations enter the linear operation. Modern TorchAO versions may choose among packing formats and external kernel libraries such as MSLK.

A weight-only conversion replaces eligible linear modules with a representation that stores packed low-bit weights and dispatches a compatible operator for floating-point inputs. The theoretical bandwidth reduction appears only if conversion succeeds, packing is retained, and the runtime avoids materializing full dequantized weights. Package metadata alone proves none of those conditions.

The compatibility chain is `Python package → PyTorch ABI → auxiliary kernel package → GPU architecture → quantization config → converted module → executed operator`. A break near the beginning prevents meaningful memory, error, or latency comparison farther down the chain.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "15-torchao-int4"
device = require_cuda()
torch.manual_seed(2026 + 15)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | BF16 linear module, reserved as the fallback path |
| Candidate | TorchAO `Int4WeightOnlyConfig` conversion on the same CUDA stack |
| Held constant | PyTorch 2.12/CUDA 13 environment, RTX 5090, layer/config intent |
| Measurements | package presence, conversion status, exact exception class/message; downstream metrics only on success |
| Evidence | `compatibility-probe` |

**Experiment:** Convert a BF16 linear layer with TorchAO INT4, record the resulting module type, compare output error, and time both paths.


## 5. Read the experiment code

The notebook attempts the documented native configuration inside an explicit compatibility boundary and records the exact failure class when the path cannot execute.

The notebook imports TorchAO, constructs the intended conversion, and catches the exact failure rather than replacing the candidate. The JSON result records both `torchao_installed=true` and `conversion=failed`, which distinguishes package discovery from backend readiness.

Because conversion stopped before a quantized module existed, the notebook correctly omits storage, output-error, operator, and latency numbers for the candidate. Fabricating those from a reference quantizer would answer a different lesson.

Only after these variables match the protocol should the cell be executed.


In [2]:
import copy, importlib.util
available=importlib.util.find_spec("torchao") is not None; details={"torchao_installed":available}
if available:
    try:
        from torchao.quantization import Int4WeightOnlyConfig, quantize_
        layer=torch.nn.Linear(4096,4096,bias=False,device=device,dtype=torch.bfloat16); candidate=copy.deepcopy(layer)
        x=torch.randn(8,4096,device=device,dtype=torch.bfloat16); ref=layer(x)
        quantize_(candidate,Int4WeightOnlyConfig(group_size=128)); out=candidate(x)
        details.update({"conversion":"success","module_type":type(candidate).__name__,"output_error":error_metrics(ref,out),
                        "bf16_timing":cuda_benchmark(lambda:layer(x),warmup=5,repeats=20),
                        "int4_timing":cuda_benchmark(lambda:candidate(x),warmup=5,repeats=20)})
    except Exception as exc:
        details.update({"conversion":"failed","error_type":type(exc).__name__,"error_message":str(exc)[:240]})
result=base_result(15,"native-backend" if details.get("conversion")=="success" else "compatibility-probe")
outcome=("TorchAO INT4 converted and executed; output error and latency were measured."
         if details.get("conversion")=="success" else
         "TorchAO was installed, but the native INT4 path did not execute; the dependency failure is preserved as a compatibility result.")
result.update({"torchao":details,"conclusion":outcome})


W0807 22:45:46.374000 451982 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| TorchAO installed | yes |
| Conversion status | failed |
| Failure type | ImportError |
| Failure message | Requires mslk >= 1.0.0 |


## 7. Interpret rather than merely print

TorchAO was present, but conversion raised `ImportError: Requires mslk >= 1.0.0`. The native INT4 operator did not execute, so the evidence label is `compatibility-probe`, not `native-backend`. This negative result establishes the exact boundary of the saved environment and a reproducible next action.

Lesson 01 used a full-model TorchAO path that did execute under its tested configuration. The contrast is valuable: backend support can depend on API/configuration and dependency versions even on the same GPU, so results must stay attached to their exact path.

**Inspection rule:** Require conversion success, storage accounting, output error, and repeated latency. A missing TorchAO install becomes an explicit compatibility result.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The named optional backend did not complete a native run in this environment. Package and failure evidence are retained; service or kernel performance is not inferred.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "TorchAO was installed, but the native INT4 path did not execute; the dependency failure is preserved as a compatibility result.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "compatibility-probe",
  "executed_at_utc": "2026-08-07T14:45:46+00:00",
  "lesson": 15,
  "schema_version": 1,
  "torchao": {
    "conversion": "failed",
    "error_message": "Requires mslk >= 1.0.0",
    "error_type": "ImportError",
    "torchao_installed": true
  }
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Treat TorchAO INT4 as a measured backend path, not a universal performance property of four-bit weights.

**Acceptance/rollback:** Require successful import/conversion, quantized tensor/module identity, storage accounting, operator evidence, output error, and repeated latency. Preserve dependency failure rather than falling back silently.

**Failure analysis:** The worst response would be to catch the error, run a hand-written fake quantizer, and leave the heading 'TorchAO benchmark'. Another failure is installing arbitrary nightly wheels until import succeeds without checking ABI compatibility or whether the environment was altered for other lessons.


## 10. Extend the evidence

Create an isolated environment using the TorchAO compatibility matrix, install a matching MSLK/PyTorch build, and rerun conversion. Only after success should the lab add module type, packed storage, output error, operator trace, warm-up, repeated latency, and a comparison with the BF16 baseline.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
